In [1]:
"""
LDA (Latent Dirichlet Allocation) 토픽 모델링 구현
Opinosis 데이터셋을 활용한 코드
"""

import os, glob, re
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from pycaret.clustering import * # setup, create_model, assign_model

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import warnings

warnings.filterwarnings("ignore")

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\july1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
def LDATopicModeling(n_topics=6, random_state=23):
    """
    Parameters:
    -----------
    n_topics : int
        추출할 토픽의 개수 (K)
    random_state : int
        재현성을 위한 랜덤 시드
    """
    n_topics = n_topics
    random_state = random_state
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words("english"))

    return lemmatizer, stop_words


# ==========================================
# 방법 2: Scikit-learn을 이용한 LDA
# ==========================================
def train_sklearn_lda(documents, n_topics=6, random_state=23):
    """
    Scikit-learn을 이용한 LDA 모델 학습

    Parameters:
    -----------
    documents : list of str
        원본 문서 리스트

    Returns:
    --------
    lda_model : sklearn.decomposition.LatentDirichletAllocation
        학습된 LDA 모델
    vectorizer : sklearn.feature_extraction.text.CountVectorizer
        벡터라이저
    dtm : sparse matrix
        문서-단어 행렬
    """
    # CountVectorizer로 DTM 생성
    vectorizer = CountVectorizer(
        max_df=0.8,
        min_df=2,
        stop_words="english",
        lowercase=True,
        token_pattern="[a-zA-Z\-][a-zA-Z\-]{2,}",
    )

    dtm = vectorizer.fit_transform(documents)

    # LDA 모델 학습
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=random_state,
        max_iter=50,
        learning_method="online",
        n_jobs=-1,
    )

    lda_model.fit(dtm)

    # Perplexity 출력
    perplexity = lda_model.perplexity(dtm)
    print(f"Scikit-learn LDA Perplexity: {perplexity:.4f}")

    return lda_model, vectorizer, dtm


def get_document_topics_sklearn(lda_model, dtm):
    """
    각 문서의 토픽 분포 추출 (Scikit-learn)

    Returns:
    --------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포 행렬 (n_documents x n_topics)
    """
    doc_topic_dist = lda_model.transform(dtm)
    return doc_topic_dist


# ==========================================
# 결과 시각화 및 분석
# ==========================================

def display_topics(model, feature_names=None, n_top_words=10, model_type="sklearn"):
    """
    각 토픽의 주요 단어 출력

    Parameters:
    -----------
    model : LDA model
        학습된 LDA 모델
    feature_names : list
        단어 리스트 (sklearn의 경우)
    n_top_words : int
        출력할 단어 개수
    model_type : str
        'gensim' 또는 'sklearn'
    """
    print(f"\n{'='*60}")
    print(f"주요 토픽 및 키워드 ({model_type.upper()})")
    print(f"{'='*60}\n")

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f"토픽 {topic_idx + 1}:")
        print(f"  {', '.join(top_words)}\n")

In [3]:
# ==========================================
# 지도 학습 : LDA 출력을 Feature로 활용
# ==========================================

def supervised_learning_example(doc_topic_dist, labels):
    """
    LDA의 문서-토픽 분포를 Feature로 사용한 분류 

    Parameters:
    -----------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포 (LDA 출력)
    labels : numpy.ndarray
        실제 레이블 (예: 감성 레이블, 제품 카테고리)
    """
    print(f"\n{'='*60}")
    print("지도 학습 : LDA Feature + Logistic Regression")
    print(f"{'='*60}\n")

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        doc_topic_dist, labels, test_size=0.2, random_state=42
    )

    # Logistic Regression 학습
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_train, y_train)

    # 예측 및 평가
    y_pred = clf.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

In [4]:
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [ ]:
document_df.head()

In [5]:
# ==========================================
# 실행 
# ==========================================
lemmatizer, stop_words = LDATopicModeling(n_topics=5, random_state=42)

print("=" * 60)
print("LDA 토픽 모델링 시작")
print("=" * 60)


LDA 토픽 모델링 시작


In [6]:
# ==========================================
# 방법 2: Scikit-learn LDA
# ==========================================
print("\n[방법 2] Scikit-learn을 이용한 LDA\n")

sklearn_model, vectorizer, dtm = train_sklearn_lda(document_df["processed_text"])
doc_topics_sklearn = get_document_topics_sklearn(sklearn_model, dtm)

feature_names = vectorizer.get_feature_names_out()
display_topics(sklearn_model, feature_names, model_type="sklearn", n_top_words=5)

print("\n문서-토픽 분포 (첫 10개 문서):")
print(doc_topics_sklearn[:10])


[방법 2] Scikit-learn을 이용한 LDA

Scikit-learn LDA Perplexity: 694.5865

주요 토픽 및 키워드 (SKLEARN)

토픽 1:
  room, location, hotel, staff, service

토픽 2:
  video, price, free, sound, camera

토픽 3:
  battery, screen, life, button, keyboard

토픽 4:
  free, location, battery, life, room

토픽 5:
  mileage, interior, car, gas, comfortable


문서-토픽 분포 (첫 10개 문서):
[[5.05240546e-04 4.99567374e-04 9.97994766e-01 4.92903138e-04
  5.07523236e-04]
 [9.98624297e-01 3.45564556e-04 3.46546501e-04 3.39224782e-04
  3.44367034e-04]
 [3.11925716e-04 3.19546296e-04 9.98747620e-01 3.08805283e-04
  3.12102993e-04]
 [4.47842150e-04 5.10142094e-01 4.88517357e-01 4.43646636e-04
  4.49060340e-04]
 [8.38648658e-05 8.42757481e-05 9.99666725e-01 8.20794019e-05
  8.30548139e-05]
 [1.74488169e-04 1.74563217e-04 9.99303941e-01 1.71928624e-04
  1.75078593e-04]
 [1.88077551e-04 1.87972465e-04 1.87386638e-02 1.84946948e-04
  9.80700339e-01]
 [2.73953163e-04 2.73349557e-04 1.67472431e-02 2.68932782e-04
  9.82436521e-01]
 [2.4128982

In [7]:
def perform_advanced_pycaret_clustering(doc_topic_dist, documents=None):
    """
    고급 PyCaret 클러스터링 - 여러 알고리즘 비교 및 최적화

    Parameters:
    -----------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포
    documents : list of str
        원본 문서

    Returns:
    --------
    results : dict
        각 모델별 결과
    """
    print("\n" + "=" * 60)
    print("고급 클러스터링 분석 - 여러 알고리즘 비교")
    print("=" * 60 + "\n")

    # DataFrame 생성
    topic_columns = [f"Topic_{i+1}" for i in range(doc_topic_dist.shape[1])]
    df = pd.DataFrame(doc_topic_dist, columns=topic_columns)
    df["Document_ID"] = range(len(df))

    # PyCaret setup
    cluster_setup = setup(
        data=df,
        session_id=42,
        normalize=True,
        transformation=False,
        ignore_features=["Document_ID"],
        verbose=False,
        html=False,
    )

    # 다양한 클러스터링 알고리즘 테스트
    algorithms = ["kmeans", "ap", "meanshift", "sc", "hclust", "dbscan"]
    # algorithms = models()
    results = {}

    print("다양한 클러스터링 알고리즘 테스트 중...\n")

    for algo in algorithms:
        try:
            print(f"Testing {algo.upper()}...")

            # 모델 생성
            if algo in ["kmeans", "sc", "hclust"]:
                model = create_model(algo, num_clusters=3)
            else:
                model = create_model(algo)

            # 예측
            predictions = assign_model(model)
            cluster_labels = predictions["Cluster"].values

            # 평가
            from sklearn.metrics import silhouette_score, davies_bouldin_score

            silhouette = silhouette_score(doc_topic_dist, cluster_labels)
            davies_bouldin = davies_bouldin_score(doc_topic_dist, cluster_labels)
            n_clusters = len(np.unique(cluster_labels))

            results[algo] = {
                "model": model,
                "predictions": predictions,
                "silhouette_score": silhouette,
                "davies_bouldin_score": davies_bouldin,
                "n_clusters": n_clusters,
            }

            print(
                f"  ✓ Clusters: {n_clusters}, Silhouette: {silhouette:.4f}, "
                f"Davies-Bouldin: {davies_bouldin:.4f}\n"
            )

        except Exception as e:
            print(f"  ✗ {algo} 실패: {str(e)[:50]}\n")
            continue

    # 결과 요약
    print("\n" + "=" * 60)
    print("알고리즘별 성능 비교 요약")
    print("=" * 60 + "\n")

    # 결과를 DataFrame으로 정리
    summary_data = []
    for algo, result in results.items():
        summary_data.append(
            {
                "Algorithm": algo.upper(),
                "N_Clusters": result["n_clusters"],
                "Silhouette": result["silhouette_score"],
                "Davies_Bouldin": result["davies_bouldin_score"],
            }
        )

    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values("Silhouette", ascending=False)
    print(summary_df.to_string(index=False))

    # 최고 성능 모델 선택 (Silhouette 기준)
    best_algo = summary_df.iloc[0]["Algorithm"].lower()
    best_model_result = results[best_algo]

    print(f"\n최고 성능 알고리즘: {best_algo.upper()}")
    print(f"  Silhouette Score: {best_model_result['silhouette_score']:.4f}")
    print(f"  클러스터 개수: {best_model_result['n_clusters']}")

    return results, best_algo, best_model_result

In [15]:
results, best_algo, best_model_result = perform_advanced_pycaret_clustering(
    doc_topics_sklearn, document_df["opinion_text"]
)


고급 클러스터링 분석 - 여러 알고리즘 비교

다양한 클러스터링 알고리즘 테스트 중...

Testing KMEANS...


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0       0.526            33.9661          0.9957            0           0   

   Completeness  
0             0  
  ✓ Clusters: 3, Silhouette: 0.7602, Davies-Bouldin: 0.5954

Testing AP...


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.6196            63.2329          0.4684            0           0   

   Completeness  
0             0  


  ✓ Clusters: 5, Silhouette: 0.9017, Davies-Bouldin: 0.2581

Testing MEANSHIFT...


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.4665            27.7062           0.934            0           0   

   Completeness  
0             0  
  ✓ Clusters: 3, Silhouette: 0.6496, Davies-Bouldin: 0.7794

Testing SC...


...

  ✓ Clusters: 3, Silhouette: 0.7707, Davies-Bouldin: 0.4131

Testing HCLUST...


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.5367            35.4875          0.7622            0           0   

   Completeness  
0             0  


  ✓ Clusters: 3, Silhouette: 0.7707, Davies-Bouldin: 0.4131

Testing DBSCAN...


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.4249            20.6256          1.0917            0           0   

   Completeness  
0             0  


  ✓ Clusters: 5, Silhouette: 0.1669, Davies-Bouldin: 1.4138


알고리즘별 성능 비교 요약

Algorithm  N_Clusters  Silhouette  Davies_Bouldin
       AP           5    0.901711        0.258067
       SC           3    0.770687        0.413121
   HCLUST           3    0.770687        0.413121
   KMEANS           3    0.760173        0.595382
MEANSHIFT           3    0.649574        0.779410
   DBSCAN           5    0.166867        1.413781

최고 성능 알고리즘: AP
  Silhouette Score: 0.9017
  클러스터 개수: 5
